In [ ]:
import sys
sys.path.append('..')
from XAIbenchmark import XAIFramework
import warnings
import openpyxl
import torch
warnings.filterwarnings("ignore")

%load_ext autoreload
%autoreload 2

from transformers import AutoModelForSequenceClassification, AutoTokenizer
from explainers import InputXGradientExplainer, IntegratedGradientsExplainer, DeepLiftExplainer, LimeExplainer, SHAPExplainer, SHAPIQExplainer, GuidedBackpropExplainer
from evaluators import AUCTPEvaluator, SoftComprehensivenessEvaluator, SoftSufficiencyEvaluator, FADEvaluator, SparsenessEvaluator, ComplexityEvaluator, IOUF1Evaluator,TokenF1Evaluator, AUPRCEvaluator
from dataset_loaders.dataset_loader import LoadDatasetArgs,load_fields_from_dataset
from dataset_loaders.movie_rationales import MovieRationalesProcessor
from dataset_loaders.hatexplain import HateSpeechProcessor

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model_name = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

ig= IntegratedGradientsExplainer(model,tokenizer)
gb= GuidedBackpropExplainer(model,tokenizer)
dl= DeepLiftExplainer(model,tokenizer)
ixg= InputXGradientExplainer(model,tokenizer,multiply_by_inputs=True)
g= InputXGradientExplainer(model,tokenizer,multiply_by_inputs=False)
lime= LimeExplainer(model,tokenizer)
shap= SHAPExplainer(model,tokenizer)
shapiq= SHAPIQExplainer(model,tokenizer)

sc= SoftComprehensivenessEvaluator(model,tokenizer)
ss= SoftSufficiencyEvaluator(model,tokenizer)
fad= FADEvaluator(model,tokenizer)
sp= SparsenessEvaluator(model,tokenizer)
cx= ComplexityEvaluator(model,tokenizer)
auctp= AUCTPEvaluator(model,tokenizer)
iou_f1= IOUF1Evaluator(model,tokenizer)
token_f1= TokenF1Evaluator(model,tokenizer)
auprc= AUPRCEvaluator(model,tokenizer)

xai_framework = XAIFramework(model, tokenizer,explainers=[dl,ig,gb,ixg,g,lime,shap,shapiq], evaluators=[sc,ss,fad,sp,cx,auctp,auprc,iou_f1,token_f1], device=device)

dataset_args_ = LoadDatasetArgs(
    dataset_name="eraser-benchmark/movie_rationales",
    text_field="review",
    label_field="label",
    rationale_field="evidences",
    dataset_split="test",
)

# Load the dataset fields
results = load_fields_from_dataset(dataset_args_)
input_texts= results['text']
labels= results['labels']
rationales= results['rationales']

mv= MovieRationalesProcessor(tokenizer)
processed_rationales= mv.process_dataset(input_texts, labels, rationales)

#Select a sub-sample if you want
input_texts_sample=input_texts
labels_sample=labels
rationale_sample= processed_rationales

xai_framework.benchmark_dataset(input_texts_sample, labels_sample,rationales=rationale_sample, save_path="../results/movie_rationales.xlsx")

In [ ]:
model_name = "Hate-speech-CNERG/bert-base-uncased-hatexplain"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)

xai_framework = XAIFramework(model, tokenizer,explainers=[dl,ig,gb,ixg,g,lime,shap,shapiq], evaluators=[sc,ss,fad,sp,cx,auctp,auprc,iou_f1,token_f1], device=device)
xai_framework = XAIFramework(model, tokenizer,explainers=[dlq], evaluators=[cx], device=device)

dataset_args_ = LoadDatasetArgs(
    dataset_name="Hate-speech-CNERG/hatexplain",
    text_field="post_tokens",
    label_field="annotators",
    rationale_field="rationales",
    dataset_split="test",
)

# Load the dataset fields
results_hate = load_fields_from_dataset(dataset_args_)

texts_hate= results_hate['text']
labels_hate= results_hate['labels']
rationales_hate= results_hate['rationales']

hs= HateSpeechProcessor()
processed_input_texts_hate, processed_labels_hate, processed_rationales_hate= hs.process_dataset(input_texts=texts_hate, labels=labels_hate, rationales=rationales_hate)

#Select a sub-sample if you want
input_texts_sample=processed_input_texts_hate[300:500]
labels_sample=processed_labels_hate[300:500]
rationale_sample= processed_rationales_hate[5:8]

xai_framework.benchmark_dataset(input_texts_sample, labels_sample, rationale_sample, save_path="../results/hate_xplain.xlsx")